# Understand the reference score and its calibration

This notebook explains how observed movement and prediction error enter the score, then replays a normal-only calibration reference without refitting the research model.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/v2_data.py').is_file()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/v2_data.py').is_file())
sys.path.insert(0, str(PROJECT))
REPOSITORY = PROJECT.parents[1]
from worldmodels.v2_data import load_cohort
from worldmodels import v2_evaluation as ev
from worldmodels import v2_models as models
from worldmodels.v2_runner import window_errors, verify_freeze
COHORT = PROJECT/'artifacts/runs/reference-v2-cohort-v1'
RUN = PROJECT/'artifacts/runs/reference-v2-core-v1'
TRANSFER = PROJECT/'artifacts/runs/reference-v2-transfer-v1'
FIGURES = PROJECT/'artifacts/executed-reference-v2/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
USED_INPUTS={}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def record_input(path):
    path=Path(path).resolve()
    key=str(path.relative_to(REPOSITORY)) if path.is_relative_to(REPOSITORY) else str(path)
    USED_INPUTS[key]=sha(path)
def read_json(path):
    path = Path(path)
    assert path.is_file(), f'Required completed artifact is absent: {path}. Read docs/reference_v2/README.md for prerequisites.'
    record_input(path)
    return json.loads(path.read_text())
def read_csv(path, **kwargs):
    record_input(path)
    return pd.read_csv(path, **kwargs)
def artifact(name, directory=RUN):
    path = directory/name
    manifest = read_json(directory/'artifact_hashes.json')
    assert name in manifest and path.is_file(), f'Unrecorded or missing artifact: {path}'
    assert sha(path) == manifest[name], f'Artifact hash differs from the completed run: {path}'
    record_input(path)
    return path
def completed_run():
    result = read_json(artifact('results.json'))
    assert result['status']=='complete' and not result.get('test_only', False)
    return result
def save_figure(figure, name):
    figure.savefig(FIGURES/(name+'.svg'), bbox_inches='tight')
    figure.savefig(FIGURES/(name+'.png'), dpi=150, bbox_inches='tight')
freeze = verify_freeze()
record_input(PROJECT/'docs/reference_v2/protocol_freeze.json')
x, valid, metadata, actual_times = load_cohort(COHORT, verify=True)
cohort_identity=read_json(COHORT/'identity.json');record_input(COHORT/'cohort.csv')
for name in cohort_identity['prepared_inputs']:
    record_input(REPOSITORY/cohort_identity['prepared_directory']/name)
assert x.shape == (len(metadata),24,16,2) and len(metadata)==2815
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
print('Verified real observations:',len(metadata),'windows;',metadata.source_id.nunique(),'source videos.')
print('Observation units: pelvis-relative 2D coordinates divided by a prefix torso scale. Times are decoder seconds.')


## Reproduce the simple reference comparison

Persistence repeats the most recent measured joint positions. Its future error describes how far the observed articulation moved away from those anchors. A negative persistence error assigns a higher anomaly score to lower observed movement. That direction was identified post hoc in the preceding analysis and is locked as a baseline in this study.

In [ ]:
result = completed_run()
source = read_csv(artifact('source_anomaly_scores.csv'))
np.testing.assert_allclose(source.low_motion,-source.persistence)
display(source[['source_id','label','persistence','low_motion','ridge_posture_velocity','calibrated_two_sided']].head())
display(result['primary_anomaly'])
print('Larger anomaly scores indicate more abnormal annotation; lower forecast MSE indicates a better forecast.')


## Construct nuisance covariates from the past

We measure relative 2D movement from adjacent observed joints divided by actual decoder time gaps. We also measure the fraction of missing prefix joints. Each quantity is averaged over windows, sequences, and sources. Neither quantity requires a future observation or a diagnosis.

In [ ]:
covariates = ev.source_covariates(x,valid,actual_times,metadata)
display(covariates.describe())
changed=x[:3].copy();changed[:,16:]=np.nan
changed_support=valid[:3].copy();changed_support[:,16:]=~changed_support[:,16:]
changed_times=actual_times[:3].copy();changed_times[:,16:]=np.nan
a=ev.window_covariates(x[:3],valid[:3],actual_times[:3])
b=ev.window_covariates(changed,changed_support,changed_times)
pd.testing.assert_frame_equal(a,b)
print('Future-value, future-support, and future-time changes leave prefix covariates unchanged.')


## Separate predictor fitting, nuisance fitting, and calibration

Each outer evaluation fold is kept separate. Within its normal training pool, one set of sources fits the predictor and nuisance reference, while another set calibrates residuals. The nuisance regression uses out-of-fit normal errors. The same final predictor scores both calibration and outer sources, so calibration sources never enter their own reference construction.

In [ ]:
schedule = ev.calibration_schedule(metadata,outer_fold=0)[0]
display({key:schedule[key] for key in ['outer_fold','rotation','predictor_fit_sources','calibration_sources','heldout_sources']})
assert ev.validate_calibration_schedule(schedule)
ledger=read_json(artifact('calibration_models.json'))
entry=next(row for row in ledger if row['outer_fold']==0 and row['rotation']==0)
calibrator=read_json(artifact(entry['calibrator_path']))
display(pd.DataFrame(calibrator['nuisance_reference_rows'])[['source_id','mse','prefix_motion','prefix_missingness']].head())
display(calibrator['nuisance_model'])


## Interpret a two-sided normal-reference score

The scalar residual asks whether a source is easier or harder to predict than expected from its prefix movement and missingness. The empirical reference distribution is fitted only from the held-out normal calibration sources. A two-sided score is large at either tail. It does not assume that every impaired movement must be more erratic.

In [ ]:
scores=read_csv(artifact(entry['scores_path']))
replayed=ev.score_source_calibrator(calibrator,scores)
np.testing.assert_allclose(replayed.two_sided,scores.two_sided,atol=1e-12,rtol=0)
reference=np.asarray(calibrator['calibration_residuals'])
grid=np.linspace(min(reference.min(),scores.residual.min())-.2,max(reference.max(),scores.residual.max())+.2,180)
fig,axes=plt.subplots(1,2,figsize=(11,3.8))
axes[0].step(grid,ev.empirical_mid_cdf(reference,grid),where='mid',color='#197c80')
axes[0].plot(reference,ev.empirical_mid_cdf(reference,reference),'o',color='#a45d21',label='Normal calibration source')
axes[0].legend(loc='lower right',fontsize=9)
axes[0].set(xlabel='Scalar log-error residual',ylabel='Smoothed empirical cumulative rank',title='Only normal calibration residuals define the reference.')
axes[1].step(grid,abs(2*ev.empirical_mid_cdf(reference,grid)-1),where='mid',color='#197c80')
axes[1].set(xlabel='Scalar log-error residual',ylabel='Two-sided anomaly score',title='Either tail can indicate an unusual observation.')
fig.tight_layout();save_figure(fig,'21_calibration_reference');plt.show()
display(scores[['source_id','label','mse','residual','cdf','two_sided']].head())


## Read the calibration limits before choosing a threshold

Seven or eight calibration sources produce coarse reference ranks. Averaging rotations improves score resolution but creates neither new independent people nor an exact coverage guarantee. The reported endpoint is ranking; this study does not derive a clinically validated operating threshold.

In [ ]:
calibration_sizes=pd.DataFrame([{'fold':r['outer_fold'],'rotation':r['rotation'],'normal_calibration_sources':len(r['calibration_sources'])} for r in ledger])
display(calibration_sizes)
print('Generic split-conformal resolution with n reference observations is 1/(n+1).')
display(pd.DataFrame({'n':[7,8,19],'rank_resolution':[1/8,1/9,1/20]}))
print('The current cross-fitted two-sided score is not claimed to have split-conformal coverage.')
producer_path=artifact('calibrated_source_scores.csv')
producer=read_csv(producer_path).sort_values('source_id').reset_index(drop=True)
alternative_parser=read_csv(producer_path,float_precision='round_trip').sort_values('source_id').reset_index(drop=True)
audit=source.sort_values('source_id').reset_index(drop=True).copy()
audit['alternative_parser']=alternative_parser.calibrated_two_sided.to_numpy()
np.testing.assert_array_equal(audit.calibrated_two_sided,producer.calibrated_two_sided)
print('Largest score change from a different CSV parser:',float(np.max(np.abs(audit.alternative_parser-audit.calibrated_two_sided))))
display({'frozen_producer_AUROC':ev.within_fold_summary(audit,'calibrated_two_sided')['pair_weighted_auroc'],
         'alternative_parser_AUROC':ev.within_fold_summary(audit,'alternative_parser')['pair_weighted_auroc']})
print('The frozen producer reads its ensemble CSV with the default pandas parser before evaluating. Tiny numerical differences can break exact empirical-rank ties, so reproductions must retain this transport path. No score rounding or primary result is changed.')
print('OOF nuisance forecasts use fewer training sources than the final rotation predictor; calibration and evaluation share that final predictor.')

# Retain the exact inputs used by this independent-kernel execution.
input_path=PROJECT/'artifacts/executed-reference-v2/inputs/21_reference_scores_and_calibration.json'
input_path.parent.mkdir(parents=True,exist_ok=True)
input_path.write_text(json.dumps({'path_base':'repository root','inputs':USED_INPUTS},indent=2)+'\n')
